# 04. Demand Forecasting (Prediksi Permintaan Stok)
**Proyek**: Solusi Prima Packaging Study Case  
**Peran**: Data Analyst  
**Tujuan**: Memprediksi permintaan produk unggulan untuk minggu depan dan menentukan batas aman stok serta titik pemesanan kembali untuk reseller.

### Alasan Pemilihan Metode Simple Moving Average (SMA) 7-Hari:
1. **Praktis dan Mudah Dipahami**: Reseller membutuhkan metode yang transparan dan mudah dihitung ulang tanpa perlu latar belakang statistik yang rumit.
2. **Efektif untuk Data Harian**: Batas waktu 7 hari (mingguan) sangat cocok untuk menangkap pola fluktuasi harian dalam satu siklus mingguan (misalnya perbedaan hari kerja vs akhir pekan).
3. **Menghaluskan Variasi Singkat**: Metode ini meredam lonjakan acak (noise) pada transaksi harian sehingga memberikan tren yang lebih stabil untuk perencanaan stok jangka pendek.

---

### Import Libraries & Load Cleaned Data

In [ ]:
import os
import sys
import logging
import pandas as pd
from typing import Tuple

# =============================================================================
# 1. LOGGING & ENVIRONMENT CONFIGURATION
# =============================================================================
# Mengonfigurasi logging standar industri agar bisa dipantau via cloud/console logs
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Mengamankan base path secara absolut agar skrip bisa dijalankan dari direktori mana pun
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
PATH_PROCESSED = os.path.normpath(os.path.join(BASE_DIR, "../data/processed/"))

# Definisi skema data (Dtype Map) sesuai kolom asli hasil inspeksi berkas CSV Anda.
# Langkah ini menghemat memori RAM dan mempercepat waktu tunggu I/O Pandas.
TRANSAKSI_DTYPE = {
    'order_id': 'string',
    'reseller_id': 'string',
    'customer_id': 'string',
    'product_id': 'string',
    'qty': 'int32',
    'harga': 'float64',
    'margin': 'float64'
}
PRODUK_DTYPE = {
    'product_id': 'string',
    'nama_produk': 'string',
    'kategori': 'category',
    'harga_satuan': 'float64',
    'stok': 'int32'
}

# =============================================================================
# 2. CORE DATA INGESTION FUNCTION (DEFENSIVE DESIGN)
# =============================================================================
def load_and_validate_forecasting_data(
    base_path: str,
    transaksi_file: str = "cleaned_data_transaksi.csv",
    produk_file: str = "cleaned_data_produk.csv"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Memuat data transaksi dan produk secara defensif disertai validasi integritas skema data.
    
    Args:
        base_path (str): Direktori absolut tempat data disimpan.
        transaksi_file (str): Nama file transaksi CSV.
        produk_file (str): Nama file produk CSV.
        
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Dataframe transaksi dan produk yang siap pakai.
        
    Raises:
        FileNotFoundError: Jika berkas CSV tidak ditemukan di direktori target.
        ValueError: Jika data kosong atau kolom kritikal hilang/berubah nama.
    """
    path_transaksi = os.path.join(base_path, transaksi_file)
    path_produk = os.path.join(base_path, produk_file)
    
    # Validasi eksistensi berkas secara proaktif sebelum memakan resource memori (Fail-Fast Principle)
    if not os.path.exists(path_transaksi):
        raise FileNotFoundError(f"Komponen Kritis Hilang: File tidak ditemukan di {path_transaksi}")
    if not os.path.exists(path_produk):
        raise FileNotFoundError(f"Komponen Kritis Hilang: File tidak ditemukan di {path_produk}")
        
    logger.info("Memulai proses memuat data dari data lake lokal...")
    
    # Ingestion dengan pemetaan tipe data statis
    df_transaksi = pd.read_csv(path_transaksi, dtype=TRANSAKSI_DTYPE)
    df_produk = pd.read_csv(path_produk, dtype=PRODUK_DTYPE)
    
    # Validasi anomali data kosong (Empty DataFrame Check)
    if df_transaksi.empty or df_produk.empty:
        raise ValueError("Data Breach/Anomaly: Salah satu berkas CSV yang dimuat kosong.")
        
    # Validasi integritas kolom wajib untuk kebutuhan visualisasi dan forecasting ke depan
    required_transaksi_cols = {'tanggal', 'product_id', 'qty'}
    if not required_transaksi_cols.issubset(df_transaksi.columns):
        missing = required_transaksi_cols - set(df_transaksi.columns)
        raise ValueError(f"Skema Rusak: Kolom penting berikut tidak ada di data transaksi: {missing}")
        
    # Rekayasa tipe data tanggal dengan error handling 'coerce' untuk menangani data korup
    df_transaksi['tanggal'] = pd.to_datetime(df_transaksi['tanggal'], errors='coerce')
    
    # Penanganan data tanggal yang gagal di-parsing (NaT) demi kestabilan kalkulasi deret waktu (Time-Series)
    if df_transaksi['tanggal'].isnull().any():
        bad_rows = df_transaksi['tanggal'].isnull().sum()
        logger.warning(f"Data Quality Notice: Ditemukan {bad_rows} baris dengan format tanggal rusak (diubah ke NaT).")
        # Menghapus baris NaT secara aman agar tidak merusak pengurutan tren waktu
        df_transaksi = df_transaksi.dropna(subset=['tanggal'])
        
    logger.info(f"Ingestion Sukses. Transaksi: {df_transaksi.shape[0]} baris, Produk: {df_produk.shape[0]} baris.")
    return df_transaksi, df_produk

# =============================================================================
# 3. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    # Eksekusi fungsi load data
    df_transaksi, df_produk = load_and_validate_forecasting_data(base_path=PATH_PROCESSED)
    print("\n[SUCCESS] df_transaksi dan df_produk sinkron dengan skema file asli dan siap digunakan.")
except Exception as e:
    logger.error(f"Pipeline Eksekusi Gagal: {str(e)}")
    # Meneruskan error agar proses otomatisasi (seperti Papermill/Airflow) tahu ada kegagalan skema
    raise e

2026-06-23 13:49:17,841 [INFO] Memulai proses memuat data dari data lake lokal...
2026-06-23 13:49:17,866 [INFO] Ingestion Sukses. Transaksi: 100 baris, Produk: 8 baris.

[SUCCESS] df_transaksi dan df_produk sinkron dengan skema file asli dan siap digunakan.


### Mencari Produk Unggulan Berdasarkan Volume Penjualan

In [ ]:
from typing import Dict, Any

# =============================================================================
# 1. CORE LOGIC FUNCTION
# =============================================================================
def identify_top_performing_product(
    df_transaksi: pd.DataFrame, 
    df_produk: pd.DataFrame
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """
    Mengidentifikasi produk dengan volume penjualan tertinggi berdasarkan kuantitas (QTY).
    Menjamin keamanan eksekusi jika dataset kosong atau terjadi kegagalan relasi antar-tabel.
    
    Args:
        df_transaksi (pd.DataFrame): Dataframe transaksi penjualan.
        df_produk (pd.DataFrame): Dataframe master katalog produk.
        
    Returns:
        Tuple[Dict[str, Any], pd.DataFrame]: 
            - Dictionary berisi metadata produk teratas (id, nama, total_qty).
            - Dataframe peringkat seluruh produk yang terurut.
            
    Raises:
        ValueError: Jika data transaksi kosong atau tidak ada produk yang valid teridentifikasi.
    """
    # 1. Validasi awal (Fail-Fast Principle)
    if df_transaksi.empty:
        raise ValueError("Pipeline Error: Dataframe transaksi kosong. Tidak dapat menghitung volume penjualan.")
    
    logger.info("Memulai kalkulasi agregasi volume penjualan produk...")
    
    # 2. Agregasi volume QTY per produk secara efisien
    df_volume = (
        df_transaksi.groupby('product_id')['qty']
        .sum()
        .reset_index()
    )
    
    # 3. Penggabungan data (Merge) secara defensif.
    # Menggunakan 'inner' merge untuk memastikan produk yang di-forecast memiliki nama produk yang valid di master katalog.
    df_ranked = df_volume.merge(
        df_produk[['product_id', 'nama_produk']], 
        on='product_id', 
        how='inner'
    )
    
    # Validasi pasca-merge untuk mendeteksi integritas data (mismatch product_id)
    if df_ranked.empty:
        raise ValueError(
            "Integrity Error: Gagal mencocokkan product_id antara data transaksi dan katalog produk. "
            "Periksa kembali sinkronisasi data master Anda."
        )
        
    # 4. Pengurutan peringkat performa produk
    df_ranked = (
        df_ranked.sort_values(by='qty', ascending=False)
        .reset_index(drop=True)
    )
    
    # 5. Ekstraksi produk teratas secara aman tanpa hardcoded indeks numerik .loc[0]
    top_row = df_ranked.iloc[0]
    
    meta_top_product = {
        "product_id": str(top_row['product_id']),
        "nama_produk": str(top_row['nama_produk']),
        "total_qty": int(top_row['qty'])
    }
    
    logger.info(
        f"Identifikasi Sukses. Produk Teratas: {meta_top_product['nama_produk']} "
        f"({meta_top_product['product_id']}) dengan total penjualan {meta_top_product['total_qty']:,} unit."
    )
    
    return meta_top_product, df_ranked

# =============================================================================
# 2. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print("=== MENCARI PRODUK DENGAN VOLUME PENJUALAN TERTINGGI ===")
    
    # Memanggil fungsi inti menggunakan variabel yang sudah dimuat dari cell sebelumnya
    top_product_meta, df_sales_volume = identify_top_performing_product(df_transaksi, df_produk)
    
    # Menetapkan variabel global untuk dikonsumsi oleh modul forecasting/visualisasi berikutnya
    top_product_id = top_product_meta["product_id"]
    top_product_name = top_product_meta["nama_produk"]
    
    print("\nPeringkat Penjualan Produk (Volume Qty):")
    display(df_sales_volume)
    print(f"\nProduk Terpilih untuk Forecast: {top_product_name} ({top_product_id})")

except Exception as e:
    logger.error(f"Komponen Pemilihan Produk Gagal: {str(e)}")
    raise e

=== MENCARI PRODUK DENGAN VOLUME PENJUALAN TERTINGGI ===
2026-06-23 13:49:17,897 [INFO] Memulai kalkulasi agregasi volume penjualan produk...
2026-06-23 13:49:17,923 [INFO] Identifikasi Sukses. Produk Teratas: Botol PET Bening 500ml (PKG-BTL-01) dengan total penjualan 94,500 unit.

Peringkat Penjualan Produk (Volume Qty):


,product_id,qty,nama_produk
0,PKG-BTL-01,94500,Botol PET Bening 500ml
1,PKG-PCH-01,77000,Standing Pouch Aluminium 250g
2,PKG-BTL-02,67500,Botol Kaca Amber 250ml
3,PKG-BOX-02,38500,Dus Produk Glossy 10x10x10
4,PKG-CAP-01,35000,Tutup Botol Ulir 28mm
5,PKG-BOX-01,34500,Karton Box Die Cut 20x15x8
6,PKG-TRY-01,29500,Tray Makanan Mika 4 Sekat
7,PKG-LBL-01,18500,Stiker Label Vinyl Tahan Air



Produk Terpilih untuk Forecast: Botol PET Bening 500ml (PKG-BTL-01)


### Menyiapkan Data Deret Waktu Harian dan Simple Moving Average

In [4]:
# %% [markdown]
# ### Time-Series Aggregation & Baseline Forecasting

# =============================================================================
# 1. CORE TIME-SERIES ENGINEERING FUNCTION
# =============================================================================
def generate_time_series_forecast_exact(
    df_transaksi: pd.DataFrame,
    target_product_id: str,
    target_product_name: str,
    window_size: int = 7
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Membangun deret waktu harian berdasarkan tanggal aktif transaksi.

    Logika kalkulasi mempertahankan formula asli (mengabaikan hari tanpa 
    penjualan untuk menghindari penurunan nilai rata-rata historis).

    Args:
        df_transaksi: Dataframe mentah berisi riwayat transaksi penjualan.
        target_product_id: ID unik produk unggulan yang dipilih untuk forecast.
        target_product_name: Nama deskriptif produk unggulan terpilih.
        window_size: Ukuran jendela hari untuk Simple Moving Average (SMA).

    Returns:
        Tuple berisi:
        - daily_sales (pd.DataFrame): Dataframe runut waktu harian teragregasi.
        - forecast_meta (Dict): Metadata hasil kalkulasi rata-rata dan total prediksi.

    Raises:
        ValueError: Jika data transaksi untuk product_id terpilih kosong.
    """
    # Isolasi data menggunakan kopian lokal untuk mencegah SettingWithCopyWarning dan efek samping global
    df_filtered = df_transaksi[df_transaksi['product_id'] == target_product_id].copy()
    
    if df_filtered.empty:
        raise ValueError(f"Data Anomaly: Tidak ditemukan riwayat transaksi untuk product_id: {target_product_id}")
        
    # Memastikan standardisasi format datetime sebelum sorting kronologis
    df_filtered['tanggal'] = pd.to_datetime(df_filtered['tanggal'])
    
    # Agregasi volume harian berdasarkan tanggal aktif yang tersedia di data lake
    df_daily = df_filtered.groupby('tanggal')['qty'].sum().reset_index()
    
    # Mengurutkan kronologi dari data terlama ke terbaru demi validitas rolling window
    df_daily = df_daily.sort_values(by='tanggal').reset_index(drop=True)
    
    # Perhitungan Simple Moving Average (SMA) dengan parameter persis seperti kode asli Anda
    target_column_sma = f'SMA_{window_size}'
    df_daily[target_column_sma] = df_daily['qty'].rolling(window=window_size, min_periods=1).mean()
    
    # Ekstraksi baris terakhir dari data aktif untuk kalkulasi prediksi minggu depan
    prediksi_harian = float(df_daily[target_column_sma].iloc[-1])
    total_prediksi_minggu_depan = prediksi_harian * 7
    
    # Metadata penampung nilai untuk transfer variabel antar-cell secara eksplisit
    forecast_meta = {
        "daily_forecast_avg": prediksi_harian,
        "weekly_forecast_total": total_prediksi_minggu_depan
    }
    
    logger.info(f"Kalkulasi Berhasil. Hasil dikembalikan ke formula asli untuk {target_product_name}.")
    return df_daily, forecast_meta

# =============================================================================
# 2. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print(f"=== MEMBUAT TIMESERIES DAN FORECAST UNTUK: {top_product_name} ===")
    
    # Menjalankan fungsi dengan jaminan hasil kembali seperti semula
    daily_sales, forecast_results = generate_time_series_forecast_exact(
        df_transaksi=df_transaksi,
        target_product_id=top_product_id,
        target_product_name=top_product_name,
        window_size=7
    )
    
    # Menyisipkan kembali nilai ke variabel global agar cell visualisasi Plotly Anda berjalan normal
    prediksi_harian_minggu_depan = forecast_results["daily_forecast_avg"]
    total_prediksi_minggu_depan = forecast_results["weekly_forecast_total"]
    
    print(f"\nRata-rata permintaan harian berdasarkan tren terakhir: {prediksi_harian_minggu_depan:.2f} unit")
    print(f"Prediksi total permintaan minggu depan (7 hari): {total_prediksi_minggu_depan:.2f} unit")

except Exception as e:
    logger.error(f"Komponen Pemodelan Deret Waktu Gagal: {str(e)}")
    raise e

=== MEMBUAT TIMESERIES DAN FORECAST UNTUK: Botol PET Bening 500ml ===
2026-06-23 13:49:17,984 [INFO] Kalkulasi Berhasil. Hasil dikembalikan ke formula asli untuk Botol PET Bening 500ml.

Rata-rata permintaan harian berdasarkan tren terakhir: 8071.43 unit
Prediksi total permintaan minggu depan (7 hari): 56500.00 unit


### 1. Analisis Volume Penjualan Produk (Peringkat Teratas)

Dari total **8 lini produk kemasan** yang tercatat, **Botol PET Bening 500ml (PKG-BTL-01)** merupakan produk dengan penjualan tertinggi (*top-performing product*) dengan total penjualan mencapai **94.500 unit**.

Jika dikorelasikan dengan hasil analisis asosiasi produk (*cross-selling*) sebelumnya, dua produk dengan volume penjualan terbesar adalah:

| Peringkat | Produk | Total Penjualan |
|-----------|---------|----------------|
| 1 | Botol PET Bening 500ml | **94.500 unit** |
| 2 | Standing Pouch Aluminium 250g | **77.000 unit** |

Temuan ini memperkuat hasil analisis *cross-selling* yang menunjukkan bahwa kedua produk tersebut merupakan penggerak utama (*key revenue drivers*) bagi para reseller. Selain itu, tingginya volume penjualan **Tutup Botol Ulir 28mm** sebesar **35.000 unit** juga konsisten dengan tingginya permintaan terhadap Botol PET sebagai produk pelengkap (*complementary product*).

---

### 2. Prediksi Permintaan dan Manajemen Stok Minggu Depan

Berdasarkan rata-rata permintaan harian (*daily demand*) sebesar **8.071,43 unit**, proyeksi kebutuhan stok untuk **Botol PET Bening 500ml** selama **7 hari ke depan** adalah sebagai berikut:

| Metrik | Nilai |
|---------|---------|
| Rata-rata Permintaan Harian | 8.071,43 unit |
| Prediksi Permintaan (7 Hari) | **56.500 unit** |
| Safety Stock (2 Hari Cadangan) | **16.143 unit** |
| Lead Time | 3 hari |
| Reorder Point (ROP) | **40.357 unit** |

#### Interpretasi Operasional

- **Prediksi Permintaan:** Reseller diperkirakan membutuhkan sekitar **56.500 unit** Botol PET Bening 500ml dalam tujuh hari mendatang.
- **Safety Stock:** Disarankan menyimpan stok pengaman minimal **16.143 unit**, setara dengan dua hari kebutuhan rata-rata, untuk mengantisipasi keterlambatan pengiriman maupun lonjakan permintaan yang tidak terduga.
- **Reorder Point (ROP):** Pemesanan ulang sebaiknya dilakukan ketika stok tersisa mencapai **40.357 unit**.

Rumus yang digunakan:

$$
\text{ROP} = (\text{Lead Time} \times \text{Daily Demand}) + \text{Safety Stock}
$$

$$
(3 \times 8.071,43) + 16.143 = 40.357
$$

---

### 3. Alasan Pemilihan Metode Forecast Sederhana

Metode *moving average* atau pendekatan berbasis rata-rata tren harian dipilih karena beberapa pertimbangan praktis berikut:

#### A. Karakteristik Industri Kemasan

Produk kemasan seperti botol PET, tutup botol, dan standing pouch umumnya memiliki pola pembelian berulang (*repeat order*) yang relatif stabil. Oleh karena itu, pendekatan rata-rata historis mampu menghasilkan estimasi jangka pendek yang cukup akurat untuk kebutuhan operasional harian.

#### B. Kemudahan Implementasi

Metode ini dapat diterapkan secara langsung oleh reseller tanpa memerlukan perangkat lunak statistik atau kemampuan analisis data yang kompleks. Perhitungan dapat dilakukan menggunakan spreadsheet sederhana sehingga lebih mudah diadopsi dalam operasional sehari-hari.

#### C. Relevan untuk Horizon Jangka Pendek

Untuk periode perencanaan pendek (7–14 hari), pendekatan berbasis rata-rata historis umumnya sudah memadai dan mampu memberikan dasar pengambilan keputusan stok yang cepat serta mudah dipahami oleh tim operasional.

---

#### Kesimpulan Bisnis

Analisis menunjukkan bahwa **Botol PET Bening 500ml** merupakan produk utama yang berkontribusi terbesar terhadap volume penjualan reseller. Dengan proyeksi permintaan sebesar **56.500 unit** pada minggu berikutnya, pengelolaan persediaan perlu difokuskan pada produk ini melalui penerapan **safety stock sebesar 16.143 unit** dan **reorder point sebesar 40.357 unit**.

Pendekatan ini membantu reseller mengurangi risiko *stockout*, menjaga tingkat layanan pelanggan, serta memastikan peluang penjualan tetap dapat dimaksimalkan selama periode permintaan tinggi.

### Menghitung Safety Stock dan Reorder Point

In [ ]:
import numpy as np

# Configurable paths untuk artifact penulisan tabel laporan produksi
OUTPUT_DIR = "../outputs/tables/"
OUTPUT_FILENAME = "inventory_forecasting_summary.csv"

# =============================================================================
# 1. CORE INVENTORY ENGINEERING FUNCTION
# =============================================================================
def calculate_inventory_metrics(
    df_daily_sales: pd.DataFrame,
    product_name: str,
    weekly_forecast_total: float,
    lead_time_days: int = 3,
    max_lead_time_days: int = 5
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """Menghitung rekomendasi Safety Stock dan Reorder Point (ROP) secara defensif.

    Args:
        df_daily_sales: Dataframe deret waktu harian hasil olahan cell sebelumnya.
        product_name: Nama deskriptif produk terpilih.
        weekly_forecast_total: Nilai total estimasi permintaan 7 hari ke depan.
        lead_time_days: Waktu tunggu pengiriman normal dari pabrik (hari). Default: 3.
        max_lead_time_days: Waktu tunggu pengiriman terlama dari pabrik (hari). Default: 5.

    Returns:
        Tuple berisi:
        - metrics (Dict): Kamus data berisi nilai integer bulat hasil perhitungan.
        - df_summary (pd.DataFrame): Dataframe ringkasan laporan manajemen inventory.

    Raises:
        ValueError: Jika data penjualan kosong atau parameter lead time tidak valid.
    """
    # Validasi Input Parameter (Fail-Fast Principle)
    if df_daily_sales.empty or 'qty' not in df_daily_sales.columns:
        raise ValueError("Pipeline Error: Dataframe penjualan harian kosong atau kolom 'qty' tidak ditemukan.")
    if lead_time_days > max_lead_time_days:
        raise ValueError("Operational Error: 'lead_time_days' tidak boleh lebih besar dari 'max_lead_time_days'.")

    logger.info(f"Memulai kalkulasi manajemen inventaris untuk produk: {product_name}...")

    # Ekstraksi Metrik Penjualan Historis Harian
    avg_daily_sales = df_daily_sales['qty'].mean()
    max_daily_sales = df_daily_sales['qty'].max()

    # Implementasi Formula Logistik Konvensional (Safety Stock & ROP)
    safety_stock = (max_daily_sales * max_lead_time_days) - (avg_daily_sales * lead_time_days)
    reorder_point = (avg_daily_sales * lead_time_days) + safety_stock

    # Pembulatan ke atas secara defensif karena unit barang fisik tidak bisa desimal (Discrete Value)
    safety_stock_rounded = int(np.ceil(safety_stock))
    reorder_point_rounded = int(np.ceil(reorder_point))
    forecast_rounded = int(np.ceil(weekly_forecast_total))

    # Struktur data penampung untuk pemrosesan/API downstream
    metrics_meta = {
        "safety_stock": safety_stock_rounded,
        "reorder_point": reorder_point_rounded,
        "weekly_forecast": forecast_rounded
    }

    # Konstruksi Dataframe Ringkasan Laporan Lini Bisnis
    df_summary = pd.DataFrame({
        'Metrik': [
            'Produk', 
            'Prediksi Permintaan 1 Minggu', 
            'Safety Stock (Stok Minimum)', 
            'Reorder Point (ROP)'
        ],
        'Nilai': [
            product_name, 
            f"{forecast_rounded:,} unit", 
            f"{safety_stock_rounded:,} unit", 
            f"{reorder_point_rounded:,} unit"
        ]
    })

    # Perbaikan Sintaks: Mengembalikan Tuple objek murni tanpa penamaan keyword langsung
    return metrics_meta, df_summary


def save_inventory_report(df_report: pd.DataFrame, output_dir: str, filename: str) -> None:
    """Menangani ekspor tabel laporan logistik ke penyimpanan lokal secara aman."""
    try:
        os.makedirs(output_dir, exist_ok=True)
        full_export_path = os.path.join(output_dir, filename)
        
        # Ekspor berkas CSV secara bersih tanpa menyertakan indeks baris Pandas
        df_report.to_csv(full_export_path, index=False)
        logger.info(f"Artifact Sukses Diekspor: {full_export_path}")
    except PermissionError:
        logger.error(
            f"I/O Error: Gagal menulis berkas ke {filename}. "
            "Pastikan file tersebut sedang tidak dibuka oleh aplikasi lain (Excel) di komputer Anda."
        )
    except Exception as e:
        logger.error(f"Gagal mengekspor tabel inventaris: {str(e)}")


# =============================================================================
# 2. PIPELINE EXECUTION GATEWAY
# =============================================================================
try:
    print("=== MENGHITUNG SAFETY STOCK DAN REORDER POINT ===")

    # Menjalankan fungsi kalkulasi inti menggunakan variable hasil transfer cell sebelumnya
    inventory_meta, inventory_summary = calculate_inventory_metrics(
        df_daily_sales=daily_sales,
        product_name=top_product_name,
        weekly_forecast_total=total_prediksi_minggu_depan,
        lead_time_days=3,
        max_lead_time_days=5
    )

    # Menampilkan ringkasan metrik di konsol output VS Code
    print(f"Rekomendasi Safety Stock (Stok Minimum): {inventory_meta['safety_stock']:,} unit")
    print(f"Rekomendasi Reorder Point (Waktu Tepat untuk Pesan Kembali): {inventory_meta['reorder_point']:,} unit")

    # Mengekspor data ke dalam folder eksternal secara defensif
    save_inventory_report(
        df_report=inventory_summary,
        output_dir=OUTPUT_DIR,
        filename=OUTPUT_FILENAME
    )

except Exception as e:
    logger.error(f"Komponen Inventory Control Pipeline Gagal: {str(e)}")
    raise e

=== MENGHITUNG SAFETY STOCK DAN REORDER POINT ===
2026-06-23 13:49:18,023 [INFO] Memulai kalkulasi manajemen inventaris untuk produk: Botol PET Bening 500ml...
Rekomendasi Safety Stock (Stok Minimum): 62,282 unit
Rekomendasi Reorder Point (Waktu Tepat untuk Pesan Kembali): 80,000 unit
2026-06-23 13:49:18,035 [INFO] Artifact Sukses Diekspor: ../outputs/tables/inventory_forecasting_summary.csv


### 1. Analisis Metrik Manajemen Inventaris

#### Safety Stock (Stok Minimum): 62.282 Unit

**Arti Angka**

Ini adalah jumlah stok penyangga (*buffer stock*) yang **wajib tersedia** di gudang *reseller* dan tidak boleh digunakan dalam kondisi operasional normal.

**Fungsi Strategis**

Angka sebesar 62.282 unit ini dihitung secara presisi oleh AI untuk melindungi *reseller* dari risiko fluktuasi pasokan, seperti keterlambatan pengiriman dari vendor utama maupun lonjakan permintaan mendadak dari konsumen retail dan UMKM di *marketplace*.

#### Reorder Point (ROP / Titik Pemesanan Ulang): 80.000 Unit

**Arti Angka**

ROP menunjukkan batas kritis persediaan di gudang. Ketika stok Botol PET Bening 500 ml turun hingga mencapai **80.000 unit**, sistem harus secara otomatis memicu (*trigger*) proses pemesanan ulang kepada *supplier* pusat.

**Analisis Operasional (*Lead Time Demand*)**

Selisih antara ROP dan *Safety Stock* adalah sebesar **17.718 unit**:

$$
80.000 - 62.282 = 17.718
$$

Angka 17.718 unit tersebut merepresentasikan estimasi jumlah produk yang akan terjual selama masa tunggu (*lead time*), yaitu periode sejak pesanan dibuat hingga barang baru diterima di gudang *reseller*.

---

### 2. Implikasi Operasional bagi Reseller

Dengan diperolehnya metrik inventaris ini, *reseller* kini memiliki panduan berbasis data yang lebih akurat untuk mengoptimalkan pengelolaan stok dan modal kerja.

#### Mencegah Kehilangan Pendapatan

Botol PET Bening 500 ml merupakan produk jangkar (*anchor product*) dengan volume penjualan tertinggi, yaitu **94.500 unit**. Kehabisan stok pada produk ini berpotensi menghentikan efek *cross-selling* terhadap produk lain, seperti *Stand Pouch* dan *Stiker Label*. Oleh karena itu, penerapan ROP pada level 80.000 unit membantu menjaga kesinambungan ekosistem penjualan tersebut.

#### Meningkatkan Efisiensi Biaya Gudang

*Reseller* tidak lagi perlu mengandalkan perkiraan atau menimbun stok secara berlebihan yang dapat menghabiskan ruang penyimpanan dan modal kerja. Dengan batas stok minimum (*Safety Stock*) dan titik pemesanan ulang (*ROP*) yang telah ditentukan secara jelas, pengelolaan persediaan menjadi lebih efisien sekaligus meningkatkan perputaran arus kas (*cash flow turnover*).

### Visualisasi Tren Penjualan dan Forecast

In [ ]:
import plotly.graph_objects as go

# =============================================================================
# 1. DATA PREPARATION & IMMUTABILITY CONTROL
# =============================================================================
# Defensive copying untuk mencegah SettingWithCopyWarning dan mutasi data global.
# Menjaga integritas data 'daily_sales' jika diakses oleh cell lain.
df_plot = daily_sales.copy()

# Standarisasi tipe data datetime untuk akurasi operasi penambahan tanggal (Timedelta)
df_plot["tanggal"] = pd.to_datetime(df_plot["tanggal"])

# Kronologi data sangat krusial untuk perhitungan Moving Average (SMA_7) dan sekuens chart
df_plot = df_plot.sort_values("tanggal")

# Plotly membaca format string ISO lebih konsisten pada sumbu X untuk representasi harian
df_plot["tanggal_plot"] = df_plot["tanggal"].dt.strftime('%Y-%m-%d')

# =============================================================================
# 2. FORECAST HORIZON GENERATION
# =============================================================================
# Ekstraksi titik terakhir data historis sebagai jangkar (anchor point) garis proyeksi
last_hist_row = df_plot.iloc[-1]
last_date = last_hist_row["tanggal"]
last_sma = last_hist_row["SMA_7"]

# Distribusi uniform: Membagi rata total target mingguan menjadi target harian
daily_forecast_val = total_prediksi_minggu_depan / 7

# Membuat rentang 7 hari ke depan setelah tanggal historis terakhir
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)

# Rekayasa struktur data koordinat X & Y untuk visualisasi.
# Titik pertama diisi oleh data historis terakhir agar garis tren tidak terputus (seamless connection).
forecast_dates = [last_date.strftime('%Y-%m-%d')] + [d.strftime('%Y-%m-%d') for d in future_dates]
forecast_values = [last_sma] + [daily_forecast_val] * 7

# =============================================================================
# 3. BRANDING & VISUAL IDENTITY CONFIGURATION
# =============================================================================
# Sistem Warna: McKinsey Tier Corporate Palette (Contra-Ready untuk visibilitas tinggi)
C = {
    "navy":      "#0B1D35",             # Keperluan teks utama / kontras tinggi
    "tick":      "#6B7B8D",             # Angka sumbu, label ringan, dan pembatas
    "footnote":  "#9DA8B5",             # Teks keterangan bawah (low emphasis)
    "blue":      "#1558B0",             # Garis utama data historis (SMA)
    "forecast":  "#2A7DE1",             # Garis putus-putus proyeksi masa depan
    "slate":     "rgba(94,108,132,0.38)", # Area bising / data aktual harian (transparan)
    "grid":      "rgba(11,29,53,0.065)",  # Garis pandu Y-axis (sangat tipis)
    "axis_line": "rgba(11,29,53,0.16)",   # Garis dasar sumbu X
    "bg":        "#FFFFFF",             # Latar belakang bersih (corporate white)
}

# Penyusunan komponen teks dinamis (meta-data injeksi)
subtitle = f"Produk: {top_product_name}  ·  Total Prediksi Minggu Depan: {int(np.ceil(total_prediksi_minggu_depan)):,} unit"
title_text = "<b>Tren Penjualan Historis & Proyeksi Permintaan</b><br><span style='font-size:11px; color:#6B7B8D;'>" + subtitle + "</span>"

# Inisialisasi objek grafik Plotly
fig = go.Figure()

# =============================================================================
# 4. PLOTLY TRACES ARCHITECTURE (DATA LAYERS)
# =============================================================================
# Layer 1: Data Aktual Harian (Garis tipis & transparan untuk meredam noise visual)
fig.add_trace(go.Scatter(
    x=df_plot["tanggal_plot"], y=df_plot["qty"],
    mode="lines", name="Penjualan Harian Aktual",
    line=dict(color=C["slate"], width=1.0),
    hovertemplate="<b>%{x|%d %b %Y}</b><br>Aktual: <b>%{y:,} unit</b><extra></extra>"
))

# Layer 2: 7-Day SMA (Garis tebal solid sebagai penunjuk arah tren utama penjualan)
fig.add_trace(go.Scatter(
    x=df_plot["tanggal_plot"], y=df_plot["SMA_7"],
    mode="lines", name="Tren Bergerak 7-Hari (SMA)",
    line=dict(color=C["blue"], width=2.5),
    hovertemplate="<b>%{x|%d %b %Y}</b><br>SMA 7-Hari: <b>%{y:,.1f} unit</b><extra></extra>"
))

# Layer 3: Proyeksi / Forecast (Garis putus-putus mengindikasikan ketidakpastian/estimasi)
fig.add_trace(go.Scatter(
    x=forecast_dates, y=forecast_values,
    mode="lines", name="Proyeksi Permintaan (Minggu Depan)",
    line=dict(color=C["forecast"], width=2.5, dash="dash"),
    hovertemplate="<b>%{x|%d %b %Y}</b> (Prediksi)<br>Rencana Muat: <b>%{y:,.0f} unit/hari</b><extra></extra>"
))

# =============================================================================
# 5. ANNOTATIONS & LAYOUT REFINEMENT
# =============================================================================
# Garis vertikal penanda batas waktu (Cut-off date antara data riil dan data prediksi)
fig.add_vline(
    x=last_date.strftime('%Y-%m-%d'),
    line=dict(color=C["tick"], width=1, dash="dot"),
    annotation_text="MULAI FORECAST ",
    annotation_position="top right",
    annotation_font=dict(size=8, color=C["tick"])
)

# Pengaturan kanvas utama sesuai standar ukuran ekspor laporan manajemen (800x450px)
fig.update_layout(
    title=dict(text=title_text, x=0.01, y=0.94, font=dict(size=14)),
    font=dict(family="Arial, Helvetica, sans-serif"),
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.01, font=dict(size=9)),
    plot_bgcolor=C["bg"], paper_bgcolor=C["bg"],
    margin=dict(l=55, r=25, t=95, b=65), # Padding disesuaikan untuk ruang judul dan footnote
    width=800, height=450
)

# Kalkulasi batas Y-axis secara dinamis: Memberi ruang kosong 15% di atas nilai tertinggi
# Mencegah trace grafik terpotong atau menabrak batas atas komponen legenda
max_y_value = max(df_plot["qty"].max(), daily_forecast_val) * 1.15

fig.update_yaxes(
    showgrid=True, gridcolor=C["grid"], gridwidth=1,
    zeroline=False, showline=False,
    title_text="Kuantitas (Unit)", title_font=dict(size=10, color=C["tick"]),
    tickfont=dict(size=10, color=C["tick"]), tickformat=",",
    range=[0, max_y_value], ticks=""
)

fig.update_xaxes(
    showgrid=False, showline=True, linecolor=C["axis_line"], linewidth=1,
    tickfont=dict(size=10, color=C["tick"]), ticks="outside", tickcolor=C["axis_line"]
)

# Catatan kaki (Footnote) untuk transparansi metodologi data kepada stakeholders
fig.add_annotation(
    x=0, y=-0.16, xref="paper", yref="paper",
    text="<i>Catatan: SMA 7-Hari dihitung dari rata-rata bergerak historis. Proyeksi menggunakan model distribusi rerata prediksi mingguan ke depan secara merata selama 7 hari.</i>",
    font=dict(size=8.5, color=C["footnote"], family="Arial, Helvetica, sans-serif"),
    showarrow=False, xanchor="left", yanchor="top", align="left"
)

# =============================================================================
# 6. AUTOMATED ARTIFACT EXPORT & RENDERING
# =============================================================================
# Operasi I/O Defensif: Membuat direktori target secara otomatis jika belum terbuat di sistem
os.makedirs("../outputs/charts/", exist_ok=True)

# Ekspor gambar statis beresolusi tinggi dengan faktor skala 1.5x (anti-blur saat di-print)
fig.write_image(
    "../outputs/charts/sales_trend_and_forecast.png",
    scale=3,
    format="png"
)

# Merender grafis interaktif langsung ke area output Notebook VS Code
fig.show()

2026-06-23 13:49:19,094 [INFO] Chromium init'ed with kwargs {}
2026-06-23 13:49:19,100 [INFO] Found chromium path: C:\Program Files\Google\Chrome\Application\chrome.exe
2026-06-23 13:49:19,104 [INFO] Temp directory created: C:\Users\Vertin\AppData\Local\Temp\tmpu86rrrwb.
2026-06-23 13:49:19,109 [INFO] Opening browser.
2026-06-23 13:49:19,113 [INFO] Temp directory created: C:\Users\Vertin\AppData\Local\Temp\tmp26l3ucdk.
2026-06-23 13:49:19,120 [INFO] Temporary directory at: C:\Users\Vertin\AppData\Local\Temp\tmp26l3ucdk
2026-06-23 13:49:20,894 [INFO] Conforming 1 to file:///C:/Users/Vertin/AppData/Local/Temp/tmpu86rrrwb/index.html
2026-06-23 13:49:25,327 [INFO] Getting tab from queue (has 1)
2026-06-23 13:49:25,330 [INFO] Got F8FD
2026-06-23 13:49:26,441 [INFO] Reloading tab F8FD before return.
2026-06-23 13:49:27,257 [INFO] Putting tab F8FD back (queue size: 0).
2026-06-23 13:49:27,264 [INFO] Waiting for all cleanups to finish.
2026-06-23 13:49:27,271 [INFO] Exiting Kaleido.
2026-06-23

### 1. Analisis Tren Penjualan Historis (Juli–Agustus)

#### Volatilitas Penjualan Harian Aktual (Garis Abu-Abu Tipis)

Penjualan harian aktual menunjukkan tingkat fluktuasi yang sangat tinggi sepanjang periode pengamatan. Beberapa titik mengalami penurunan drastis hingga mendekati nol, terutama pada pertengahan Juli dan awal Agustus. Penurunan ekstrem yang bersifat sementara ini mengindikasikan kemungkinan terjadinya *stockout* (kehabisan stok) di tingkat *reseller* atau kendala distribusi operasional, bukan penurunan minat pasar secara fundamental.

Di sisi lain, grafik juga memperlihatkan lonjakan permintaan (*peak demand*) yang signifikan pada akhir Juli dengan volume penjualan melampaui **15.000 unit dalam satu hari**. Kondisi ini menunjukkan bahwa permintaan pasar terhadap produk Botol PET Bening 500 ml dapat meningkat secara tiba-tiba dan membutuhkan kesiapan pasokan yang memadai.

#### Pola Pertumbuhan Berkelanjutan (Garis Biru Tua – SMA 7 Hari)

Indikator *Simple Moving Average* (SMA) 7 hari digunakan untuk meredam volatilitas harian sehingga tren permintaan yang sebenarnya dapat terlihat dengan lebih jelas.

Hasil pengamatan menunjukkan tren pertumbuhan yang konsisten sepanjang periode Juli hingga pertengahan Agustus. Rata-rata penjualan bergerak dari level di bawah 5.000 unit per hari menuju kisaran 8.000–9.000 unit per hari. Kenaikan yang berkelanjutan ini mengindikasikan bahwa pasar produk Botol PET Bening 500 ml terus berkembang dan memiliki fundamental permintaan yang kuat.

---

### 2. Analisis Proyeksi Permintaan (Garis Putus-Putus Biru Muda)

#### Kestabilan Forecast Jangka Pendek

Setelah melewati batas vertikal *Mulai Forecast* pada 17 Agustus, model memproyeksikan permintaan dalam kondisi relatif stabil selama tujuh hari berikutnya.

Total permintaan yang diperkirakan untuk periode tersebut mencapai **56.500 unit**, atau setara dengan rata-rata sekitar **8.071 unit per hari**. Proyeksi yang cenderung mendatar menunjukkan bahwa model belum mendeteksi adanya perubahan pola permintaan yang signifikan dalam jangka pendek.

#### Alasan Pendekatan Distribusi Rata-Rata

Sesuai metodologi yang digunakan pada grafik, total prediksi mingguan didistribusikan secara merata ke dalam tujuh hari ke depan. Pendekatan ini dipilih karena memberikan target pasokan yang stabil dan mudah diimplementasikan dalam perencanaan operasional mingguan.

Bagi *reseller*, proyeksi yang stabil membantu proses pengadaan, pengelolaan gudang, dan penjadwalan distribusi sehingga risiko kekurangan maupun kelebihan stok dapat diminimalkan.

---

### 3. Implikasi terhadap Manajemen Persediaan

Grafik historis dan hasil peramalan memberikan justifikasi yang kuat terhadap rekomendasi *Safety Stock* dan *Reorder Point* (ROP) yang telah dihitung sebelumnya.

Lonjakan penjualan aktual yang pernah mencapai lebih dari **15.000 unit dalam satu hari** menunjukkan bahwa permintaan dapat meningkat secara signifikan dalam waktu singkat. Oleh karena itu, rekomendasi **Safety Stock sebesar 62.282 unit** dinilai memadai untuk menjaga kontinuitas operasional apabila terjadi lonjakan permintaan serupa atau gangguan pasokan dari pemasok utama.

Selain itu, rekomendasi **Reorder Point sebesar 80.000 unit** memastikan bahwa proses pemesanan ulang dilakukan sebelum persediaan memasuki level kritis. Dengan mekanisme ini, *reseller* dapat mempertahankan ketersediaan produk di pasar sekaligus menjaga tren pertumbuhan penjualan yang telah terbentuk selama periode observasi.
